In [1]:
!pip install transformers==4.44.2 joblib==1.4.2 scikit-learn==1.6.0 numpy==1.26.4 pandas==2.2.3 scipy==1.13.1 seaborn==0.13.2 tqdm==4.66.5 lightgbm==4.5.0 xgboost==2.1.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 76.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 100.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 47.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 MB 11.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.1
    Un

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
# Load datasets
train_df = pd.read_csv('/kaggle/input/dataset-rrck/Train_RRCK.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/dataset-rrck/Test_RRCK.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [4]:
tokenizer = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
model = AutoModelForSequenceClassification.from_pretrained('seyonec/ChemBERTa-zinc-base-v1', num_labels=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Custom dataset class
class SMILESDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=325):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        smiles = self.dataframe.iloc[idx]['SMILES']
        permeability = self.dataframe.iloc[idx]['Permeability']
        inputs = self.tokenizer(smiles, return_tensors='pt', padding="max_length", truncation=True, max_length=self.max_length)
        
        input_ids = inputs['input_ids'].squeeze(0)  # Shape: (sequence_length,)
        attention_mask = inputs['attention_mask'].squeeze(0)  # Shape: (sequence_length,)
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(permeability, dtype=torch.float)
        }


# datasets
train_dataset = SMILESDataset(train_df, tokenizer)
test_dataset = SMILESDataset(test_df, tokenizer)
batch_size = 16
# data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at seyonec/ChemBERTa-zinc-base-v1 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_epochs = 20

In [6]:
# Training loop
from tqdm import tqdm
for epoch in range(num_epochs):
    print(f"Entered Epoch {epoch + 1}")
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f'Training Epoch {epoch + 1}/{num_epochs}', unit='batch'):
        optimizer.zero_grad()

        # Move all batch tensors to device
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"].unsqueeze(1)  # still shape: (batch_size, 1)

        # Forward pass
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=labels
        )
        loss = outputs.loss
        train_loss += loss.item()

        # Backprop and optimizer step
        loss.backward()
        optimizer.step()

    avg_train_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}')

Entered Epoch 1


Training Epoch 1/20: 100%|██████████| 9/9 [00:04<00:00,  1.89batch/s]


Epoch 1/20 - Train Loss: 10.2151
Entered Epoch 2


Training Epoch 2/20: 100%|██████████| 9/9 [00:03<00:00,  2.50batch/s]


Epoch 2/20 - Train Loss: 0.4747
Entered Epoch 3


...

Epoch 18/20 - Train Loss: 0.1826
Entered Epoch 19


Training Epoch 19/20: 100%|██████████| 9/9 [00:04<00:00,  2.16batch/s]


Epoch 19/20 - Train Loss: 0.1503
Entered Epoch 20


Training Epoch 20/20: 100%|██████████| 9/9 [00:04<00:00,  2.13batch/s]

Epoch 20/20 - Train Loss: 0.1222


In [7]:
model_name = 'ChemBERTa_model_1_rrck'
model_save_path = f'/kaggle/working/{model_name}'
os.makedirs(model_save_path, exist_ok=True)

tokenizer.save_pretrained(model_save_path)
model.save_pretrained(model_save_path)

print(f'Model and tokenizer saved to {model_save_path}')

Model and tokenizer saved to /kaggle/working/ChemBERTa_model_1_rrck


In [8]:
from scipy.stats import pearsonr, spearmanr

model.eval()
test_loss = 0
test_true_labels = []
predictions = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing', unit='batch'):
      
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].unsqueeze(1).to(device).float()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        test_loss += loss.item()

        test_true_labels.extend(labels.cpu().numpy())
        preds = outputs.logits.squeeze().cpu().numpy()  
        predictions.extend(preds)

# Final test loss
avg_test_loss = test_loss / len(test_loader)
print(f'Test Loss: {avg_test_loss:.4f}')

test_true_labels = np.array(test_true_labels).flatten()
predictions = np.array(predictions)
print(test_true_labels.shape)
print(predictions.shape)

# Performance metrics
mse = mean_squared_error(test_true_labels, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(test_true_labels, predictions)
r2 = r2_score(test_true_labels, predictions)
PCC,_ = pearsonr(test_true_labels, predictions)
SCC,_ = spearmanr(test_true_labels, predictions)
# Print performance metrics
print(f'Mean Squared Error: {mse:.4f}')
print(f'Root Mean Squared Error: {rmse:.4f}')
print(f'Mean Absolute Error: {mae:.4f}')
print(f'R^2 Score: {r2:.4f}')
print(f'Pearson Correlation Coefficient: {PCC:.4f}')
print(f'Spearman Correlation Coefficient: {SCC:.4f}')

# Print hyperparameters
print("Hyperparameters:")
print(f"Learning Rate: {5e-5}")
print(f"Batch Size: 16")
print(f"Epochs: {num_epochs}")

Testing: 100%|██████████| 3/3 [00:00<00:00,  8.87batch/s]

Test Loss: 0.4020
(35,)
(35,)
Mean Squared Error: 0.4237
Root Mean Squared Error: 0.6510
Mean Absolute Error: 0.5062
R^2 Score: 0.0826
Pearson Correlation Coefficient: 0.5482
Spearman Correlation Coefficient: 0.4806
Hyperparameters:
Learning Rate: 5e-05
Batch Size: 16
Epochs: 20


In [9]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model_name = 'ChemBERTa_model_1_rrck'
model_save_path = f'/kaggle/working/{model_name}'

if not os.path.exists(model_save_path):
    raise FileNotFoundError(f"The model directory {model_save_path} does not exist.")

tokenizer = AutoTokenizer.from_pretrained(model_save_path)
model = AutoModel.from_pretrained(model_save_path).to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at /kaggle/working/ChemBERTa_model_1_rrck and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# Load your datasets
train_df = pd.read_csv('/kaggle/input/dataset-rrck/Train_RRCK.csv')
train_df = train_df[['ID', 'SMILES', 'Permeability']]
test_df = pd.read_csv('/kaggle/input/dataset-rrck/Test_RRCK.csv')
test_df = test_df[['ID', 'SMILES', 'Permeability']]

In [11]:
train_encodings = tokenizer(list(train_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")
test_encodings = tokenizer(list(test_df['SMILES']), truncation=True, padding=True, max_length=325, return_tensors="pt")

In [12]:
from tqdm import tqdm 
batch_size = 16 

def generate_embeddings(encodings, batch_size):
    embeddings = []
    model.eval() 
    with torch.no_grad():
        for i in tqdm(range(0, len(encodings['input_ids']), batch_size), desc="Processing batches"):
            batch = {key: val[i:i + batch_size].to(device) for key, val in encodings.items()}  
            outputs = model(**batch)
            embeddings.append(outputs.last_hidden_state)
    return torch.cat(embeddings, dim=0)

In [13]:
train_embeddings = generate_embeddings(train_encodings, batch_size)
print(train_embeddings.shape)
train_embeddings = torch.mean(train_embeddings, dim=1)
print(train_embeddings.shape)

Processing batches: 100%|██████████| 9/9 [00:00<00:00, 11.70it/s]


torch.Size([140, 188, 768])
torch.Size([140, 768])


In [14]:
column_names = [f'x_fine_emb_ChemBerta{i}' for i in range(train_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=train_embeddings.cpu().numpy(), columns=column_names)
train_data = pd.concat([train_df, embeddings_df], axis=1)

In [15]:
test_embeddings = generate_embeddings(test_encodings, batch_size)
print(test_embeddings.shape)
test_embeddings = torch.mean(test_embeddings, dim=1)
print(test_embeddings.shape)

Processing batches: 100%|██████████| 3/3 [00:00<00:00, 15.58it/s]

torch.Size([35, 190, 768])
torch.Size([35, 768])


In [16]:
column_names = [f'x_fine_emb_ChemBerta{i}' for i in range(test_embeddings.shape[1])]
embeddings_df = pd.DataFrame(data=test_embeddings.cpu().numpy(), columns=column_names)
test_data = pd.concat([test_df, embeddings_df], axis=1)

In [17]:
train_data.to_csv("/kaggle/working/Train_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_rrck.csv",index=False)
test_data.to_csv("/kaggle/working/Test_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_rrck.csv",index=False)

In [18]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [19]:
train_data = pd.read_csv("/kaggle/working/Train_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_rrck.csv")
test_data = pd.read_csv("/kaggle/working/Test_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_rrck.csv")

In [20]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -4.0)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [21]:
X_train = train_data.drop(['ID','SMILES','Permeability'],axis=1)
y_train = train_data['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")

X_test = test_data.drop(['ID','SMILES','Permeability'],axis=1)
y_test = test_data['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (140, 768)
y_train shape:  (140,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (35, 768)
y_test shape:  (35,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002885 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 29952
[LightGBM] [Info] Number of data points in the train set: 112, number of used features: 768
[LightGBM] [Info] Start training from score -5.682366
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1162,0.2548,0.3408,0.7032,0.8389,0.8543,0.3182,0.4138,0.5641,0.3110,0.6013,0.5280
DecisionTreeRegressor,0.1817,0.3221,0.4263,0.5358,0.7597,0.7689,0.3498,0.4396,0.5914,0.2427,0.5553,0.4908
RandomForestRegressor,0.1090,0.2408,0.3301,0.7215,0.8494,0.8648,0.3378,0.4294,0.5812,0.2687,0.5695,0.4556
GradientBoostingRegressor,0.1211,0.2434,0.3480,0.6906,0.8325,0.8510,0.3226,0.4157,0.5680,0.3015,0.5880,0.5025
AdaBoostRegressor,0.1279,0.2646,0.3577,0.6732,0.8216,0.8405,0.3156,0.4197,0.5618,0.3166,0.6040,0.5376
XGBRegressor,0.1347,0.2699,0.3670,0.6559,0.8172,0.8384,0.3475,0.4402,0.5895,0.2476,0.5668,0.4407
ExtraTreesRegressor,0.1208,0.2525,0.3475,0.6915,0.8324,0.8440,0.3387,0.4253,0.5819,0.2668,0.5658,0.4434
LinearRegression,1.1700,0.8188,1.0817,-1.9893,0.3750,0.3518,0.6201,0.5874,0.7875,-0.3425,0.4055,0.4369
KNeighborsRegressor,0.1234,0.2642,0.3513,0.6847,0.8289,0.8346,0.3277,0.4223,0.5725,0.2905,0.5839,0.5518
SVR,0.1142,0.2428,0.3380,0.7082,0.8416,0.8510,0.3292,0.4126,0.5738,0.2872,0.5730,0.5196


In [22]:
result_df

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.1162,0.2548,0.3408,0.7032,0.8389,0.8543,0.3182,0.4138,0.5641,0.3110,0.6013,0.5280
DecisionTreeRegressor,0.1817,0.3221,0.4263,0.5358,0.7597,0.7689,0.3498,0.4396,0.5914,0.2427,0.5553,0.4908
RandomForestRegressor,0.1090,0.2408,0.3301,0.7215,0.8494,0.8648,0.3378,0.4294,0.5812,0.2687,0.5695,0.4556
GradientBoostingRegressor,0.1211,0.2434,0.3480,0.6906,0.8325,0.8510,0.3226,0.4157,0.5680,0.3015,0.5880,0.5025
AdaBoostRegressor,0.1279,0.2646,0.3577,0.6732,0.8216,0.8405,0.3156,0.4197,0.5618,0.3166,0.6040,0.5376
XGBRegressor,0.1347,0.2699,0.3670,0.6559,0.8172,0.8384,0.3475,0.4402,0.5895,0.2476,0.5668,0.4407
ExtraTreesRegressor,0.1208,0.2525,0.3475,0.6915,0.8324,0.8440,0.3387,0.4253,0.5819,0.2668,0.5658,0.4434
LinearRegression,1.1700,0.8188,1.0817,-1.9893,0.3750,0.3518,0.6201,0.5874,0.7875,-0.3425,0.4055,0.4369
KNeighborsRegressor,0.1234,0.2642,0.3513,0.6847,0.8289,0.8346,0.3277,0.4223,0.5725,0.2905,0.5839,0.5518
SVR,0.1142,0.2428,0.3380,0.7082,0.8416,0.8510,0.3292,0.4126,0.5738,0.2872,0.5730,0.5196


In [23]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.1993172389842615, -5.149306213419454, -5.5...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.03228311527216, -5.892547041954304, -6.56...","[-6.209580025896716, -5.91845532266724, -6.485...","[0.11720698171915987, 0.062082472775380954, 0...."
1,DecisionTreeRegressor,"[-5.87, -5.35, -5.805, -5.85, -5.24, -5.92, -5...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.45, -5.76, -5.4, -6.42, -7.0, -5.805, -6....","[-6.176, -5.781999999999999, -6.40799999999999...","[0.5566183611775666, 0.044000000000000136, 0.5..."
2,RandomForestRegressor,"[-6.240099999999996, -5.203100000000003, -5.42...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.108349999999997, -5.871349999999996, -6.6...","[-6.166559999999998, -5.917469999999996, -6.56...","[0.08245222495481876, 0.06679168061967183, 0.2..."
3,GradientBoostingRegressor,"[-6.224527033268661, -5.138471762747683, -5.51...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.976595576251903, -5.818863913567692, -6.5...","[-6.09729518729629, -5.838059685411119, -6.531...","[0.22115790372104407, 0.06749603564943793, 0.3..."
4,AdaBoostRegressor,"[-6.2125, -5.25037037037037, -5.46375000000000...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.042916666666668, -5.991818181818181, -6.8...","[-6.121050163800165, -5.903422244422244, -6.75...","[0.17910266660321605, 0.1078472420086206, 0.17..."
5,XGBRegressor,"[-6.206019, -5.330262, -5.6836796, -6.059387, ...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.968209, -5.768321, -6.926013, -6.4685745,...","[-6.1250052, -5.8866677, -6.6933594, -6.351722...","[0.20771961, 0.12202537, 0.35376403, 0.4111425..."
6,ExtraTreesRegressor,"[-6.3591499999999925, -5.201500000000003, -5.4...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.054199999999996, -5.7789999999999955, -6....","[-6.113019999999998, -5.822619999999997, -6.46...","[0.07905125931950749, 0.053817930097693575, 0...."
7,LinearRegression,"[-5.887113926637571, -5.6286830931013005, -6.2...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-4.746595280028235, -5.941767155659754, -6.2...","[-5.358464875409998, -5.790215270661514, -5.93...","[1.0731439662990963, 0.25804866207758387, 0.15..."
8,KNeighborsRegressor,"[-6.260000000000001, -5.18, -5.18, -5.74333333...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-5.919999999999999, -5.919999999999999, -5.9...","[-6.127999999999999, -6.068, -6.13066666666666...","[0.11334509742865441, 0.1273071526313868, 0.25..."
9,SVR,"[-6.282734980109963, -5.2389757187738315, -5.4...",0 -6.340 1 -5.950 2 -6.240 3 -7.00...,"[[-6.054867975606758, -5.883389148936506, -6.3...","[-6.10237258687124, -5.905078758726899, -6.353...","[0.040847731405697, 0.04823036969164491, 0.349..."


In [24]:
result_df.to_csv('/kaggle/working/Results_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_rrck.csv')
prediction_df.to_csv('/kaggle/working/Prediction_data_ChemBERTa-zinc-base-v1_fine_tuned_embeddings_rrck.csv')